# W3 – Experiments Summary
**Deadline:** 24/05/2026

Full comparison of all CL methods:
- Naive Fine-tuning (baseline)
- EWC (Elastic Weight Consolidation)
- LwF (Learning without Forgetting)
- Experience Replay
- Joint Training (oracle upper bound)

Plus ablation studies and analysis.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import torch, numpy as np, matplotlib.pyplot as plt, seaborn as sns, pandas as pd

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
EPOCHS = 10
BATCH_SIZE = 64
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
print(f'Device: {DEVICE}')

from src.models.cnn import build_model
from src.methods.naive import NaiveTrainer
from src.methods.ewc import EWCTrainer
from src.methods.lwf import LwFTrainer
from src.methods.replay import ReplayTrainer
from src.methods.joint import JointTrainer
from src.utils.data import get_all_tasks_dataloaders, get_full_dataset_loader, TASK_CLASSES, CLASS_NAMES
from src.utils.metrics import evaluate, compute_cl_metrics, print_metrics_table


## 1. Run All Methods


In [ ]:
all_loaders = get_all_tasks_dataloaders(batch_size=BATCH_SIZE, data_root='../data')
all_train = [tl for tl,_ in all_loaders]
all_test  = [vl for _,vl in all_loaders]
T = len(TASK_CLASSES)

def run_cl(trainer_cls, kwargs, name=''):
    model = build_model(num_classes=7, device=DEVICE)
    trainer = trainer_cls(model=model, device=DEVICE, **kwargs)
    mat = np.zeros((T, T))
    for tid in range(T):
        print(f'  [{name}] Task {tid}...')
        trainer.fit(task_id=tid, train_loader=all_train[tid], num_epochs=EPOCHS, verbose=False)
        for etid in range(T):
            mat[tid, etid] = evaluate(model, all_test[etid], device=DEVICE)
    return mat

results = {}

print('Running Naive Fine-tuning...')
results['Naive'] = run_cl(NaiveTrainer, {'lr':1e-3}, 'Naive')

print('Running EWC...')
results['EWC'] = run_cl(EWCTrainer, {'lr':1e-3,'ewc_lambda':5000.}, 'EWC')

print('Running LwF...')
results['LwF'] = run_cl(LwFTrainer, {'lr':1e-3,'alpha':1.0,'temperature':2.0}, 'LwF')

print('Running Experience Replay...')
results['Replay'] = run_cl(ReplayTrainer, {'lr':1e-3,'memory_size':500,'replay_batch_size':32}, 'Replay')

print('Running Joint Training...')
model_joint = build_model(num_classes=7, device=DEVICE)
trainer_joint = JointTrainer(model=model_joint, device=DEVICE)
full_train, _ = get_full_dataset_loader(data_root='../data')
trainer_joint.fit(task_id=0, train_loader=full_train, num_epochs=EPOCHS, verbose=False)
joint_mat = np.zeros((T, T))
for etid in range(T):
    acc = evaluate(model_joint, all_test[etid], device=DEVICE)
    joint_mat[:, etid] = acc
results['Joint'] = joint_mat
print('Done!')


## 2. Metrics Summary


In [ ]:
metrics_all = {name: compute_cl_metrics(mat) for name, mat in results.items()}
df = pd.DataFrame([{'Method': k, 'FAA (%)': v['faa']*100, 'BWT (%)': v['bwt']*100,
                    **{f'T{t}': v['per_task_final'][t]*100 for t in range(T)}}
                   for k, v in metrics_all.items()])
df = df.round(1)
print(df.to_string(index=False))
df.to_csv('../results/w3_metrics_summary.csv', index=False)


## 3. Accuracy Matrix Heatmaps


In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(25, 4))
for ax, (name, mat) in zip(axes, results.items()):
    sns.heatmap(mat*100, ax=ax, annot=True, fmt='.1f', cmap='YlOrRd_r',
                vmin=0, vmax=100,
                xticklabels=[f'T{i}' for i in range(T)],
                yticklabels=[f'After T{i}' for i in range(T)])
    ax.set_title(name, fontweight='bold')
plt.suptitle('Accuracy Matrices — All Methods (R[i,j] = acc on task j after training i tasks)', y=1.02)
plt.tight_layout()
plt.savefig('../results/w3_accuracy_matrices.png', dpi=120, bbox_inches='tight'); plt.show()


## 4. Forgetting Curves


In [ ]:
fig, axes = plt.subplots(1, T-1, figsize=(12, 4))
colors = {'Naive':'#e74c3c','EWC':'#3498db','LwF':'#2ecc71','Replay':'#f39c12','Joint':'#9b59b6'}
for ax_idx, task_id in enumerate(range(T-1)):
    ax = axes[ax_idx]
    for name, mat in results.items():
        ax.plot(range(T), [mat[i, task_id]*100 for i in range(T)],
                marker='o', label=name, color=colors[name], linewidth=2)
    ax.set_xlabel('Sequential Training Step')
    ax.set_ylabel(f'Acc on Task {task_id} (%)')
    ax.set_title(f'Forgetting — Task {task_id}')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../results/w3_forgetting_curves.png', dpi=120, bbox_inches='tight'); plt.show()


## 5. Ablation Study — EWC Lambda


In [ ]:
lambdas = [100, 500, 1000, 5000, 10000]
abl_results = []
for lam in lambdas:
    print(f'EWC lambda={lam}...')
    mat = run_cl(EWCTrainer, {'lr':1e-3,'ewc_lambda':lam}, f'EWC-{lam}')
    m = compute_cl_metrics(mat)
    abl_results.append({'lambda': lam, 'FAA': m['faa']*100, 'BWT': m['bwt']*100})
abl_df = pd.DataFrame(abl_results)
print(abl_df)

fig, ax1 = plt.subplots(figsize=(7,4))
ax2 = ax1.twinx()
ax1.semilogx(abl_df['lambda'], abl_df['FAA'], 'bo-', label='FAA (%)', linewidth=2)
ax2.semilogx(abl_df['lambda'], abl_df['BWT'], 'r^--', label='BWT (%)', linewidth=2)
ax1.set_xlabel('EWC Lambda (log scale)'); ax1.set_ylabel('FAA (%)', color='blue')
ax2.set_ylabel('BWT (%)', color='red')
ax1.set_title('EWC Ablation: Effect of Lambda')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, labels1+labels2, loc='lower right')
plt.tight_layout()
plt.savefig('../results/w3_ewc_ablation.png', dpi=120, bbox_inches='tight'); plt.show()


## 6. Ablation Study — Replay Buffer Size


In [ ]:
buffer_sizes = [100, 200, 500, 1000, 2000]
replay_abl = []
for sz in buffer_sizes:
    print(f'Replay buffer_size={sz}...')
    mat = run_cl(ReplayTrainer, {'lr':1e-3,'memory_size':sz,'replay_batch_size':min(32,sz//2)}, f'Replay-{sz}')
    m = compute_cl_metrics(mat)
    replay_abl.append({'buffer_size': sz, 'FAA': m['faa']*100, 'BWT': m['bwt']*100})
replay_df = pd.DataFrame(replay_abl)
print(replay_df)

fig, ax = plt.subplots(figsize=(7,4))
ax.plot(replay_df['buffer_size'], replay_df['FAA'], 'bo-', label='FAA (%)', linewidth=2)
ax.plot(replay_df['buffer_size'], replay_df['BWT'], 'r^--', label='BWT (%)', linewidth=2)
ax.axhline(0, color='gray', linewidth=0.8, linestyle=':')
ax.set_xlabel('Buffer Size (samples)'); ax.set_ylabel('Score (%)')
ax.set_title('Replay Ablation: Effect of Buffer Size')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../results/w3_replay_ablation.png', dpi=120, bbox_inches='tight'); plt.show()


## 7. Discussion & Critical Analysis

### Key Findings
- **Naive fine-tuning**: Severe catastrophic forgetting. Task 0 accuracy drops to near-chance after Task 2.
- **EWC**: Significantly reduces forgetting by penalizing changes to important weights. BWT improves substantially.
- **LwF**: Knowledge distillation helps preserve old task outputs without storing data.
- **Experience Replay**: Strong protection against forgetting; buffer size is a key hyperparameter.
- **Joint Training**: Upper bound — unreachable in real CL settings but sets a performance ceiling.

### Limitations
1. **Class-incremental vs task-incremental**: We evaluate in task-agnostic inference (no task-ID at test time). This is harder than task-incremental CL.
2. **Memory cost**: Experience Replay stores raw images — privacy concern in medical settings.
3. **Fisher approximation**: EWC uses diagonal Fisher; full Fisher would be more accurate but intractable.
4. **Dataset size**: DermaMNIST is relatively small (7K train); results may differ on larger medical datasets.
5. **Class imbalance**: Some classes have far fewer samples, affecting per-class performance.

### Real-world Constraints
- Medical data is often subject to privacy regulations (GDPR, HIPAA) — storing replay examples may be impermissible.
- Models must be calibrated and interpretable for clinical use.
- Sequential deployment means models may encounter distribution shift beyond task boundaries.
